#Design the Silver layer

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("bronze_schema", "lab8_dev_bronze")
dbutils.widgets.text("bronze_table", "")
dbutils.widgets.text("silver_schema", "lab8_dev_silver")
dbutils.widgets.text("silver_table", "")
dbutils.widgets.text("key_columns", "")

In [0]:

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table_name = dbutils.widgets.get("bronze_table")
silver_schema = dbutils.widgets.get("silver_schema")
silver_table_name = dbutils.widgets.get("silver_table")

key_columns = [
    c.strip()
    for c in dbutils.widgets.get("key_columns").split(",")
]

assert bronze_table_name, "Missing bronze_table parameter"
assert silver_table_name, "Missing silver_table parameter"
assert all(key_columns), "Missing key_columns parameter"

bronze_table = f"{catalog}.{bronze_schema}.{bronze_table_name}"
silver_table = f"{catalog}.{silver_schema}.{silver_table_name}"

In [0]:
#read bronze
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_df = spark.table(bronze_table)

In [0]:
assert all(key_columns), "Specify key_columns"

for key in key_columns:
    assert key in bronze_df.columns, f"Missing key column: {key}"
    assert bronze_df.filter(
        F.col(key).isNull()
    ).count() == 0, f"NULL values in key: {key}"

In [0]:
#dedup
silver_clean_df = (bronze_df.dropDuplicates(key_columns).withColumn("silver_load_timestamp", F.current_timestamp()))

In [0]:
#DQ rules
if bronze_table_name == "environment":
    rules = (
        F.col("AREA").isNotNull()
        & F.col("YEAR").isNotNull()
        & (F.col("YEAR") >= 1900)
        & (F.col("freshwater_per_capita").isNull()
           | (F.col("freshwater_per_capita") >= 0))
        & (F.col("freshwater_withdrawals").isNull()
           | (F.col("freshwater_withdrawals") >= 0))
        & (F.col("agricultural_land").isNull()
           | (F.col("agricultural_land") >= 0))
        & (F.col("forest_area").isNull()
           | (F.col("forest_area") >= 0))
    )

    invalid_count = silver_clean_df.filter(
        ~F.coalesce(rules, F.lit(False))
    ).count()

    assert invalid_count == 0, (
        f"Data quality failed: {invalid_count} invalid rows"
    )

In [ ]:
merge_condition = " AND ".join([f"target.{c} = source.{c}" for c in key_columns])

In [0]:

# Initialize the Silver Delta table on the first run
if not spark.catalog.tableExists(silver_table):
    (
        silver_clean_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(silver_table)
    )

In [0]:
before_count = spark.table(silver_table).count()
silver_delta = DeltaTable.forName(spark, silver_table)

(
    silver_delta.alias("target")
    .merge(silver_clean_df.alias("source"), merge_condition)
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

after_count = spark.table(silver_table).count()

print("Rows before:", before_count)
print("Rows after:", after_count)

In [0]:
#check for duplicates 
duplicate_count = (
    spark.table(silver_table)
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate keys:", duplicate_count)  

In [0]:
if duplicate_count > 0:
    raise ValueError(
        f"Data quality check failed: {duplicate_count} duplicate key groups found"
    )